# 06 - Simulazione con rumore ed esecuzione su hardware reale

Questo notebook copre le due modalità di esecuzione più realistiche descritte nella sezione "Setup di esecuzione" (Capitolo 3) e implementate in `src/execution/backend_manager.py`:

1. **Simulazione con modello di rumore** (`mode="noisy_simulation"`): il modello di rumore è costruito a partire dai dati di calibrazione pubblicati per un backend reale tramite `NoiseModel.from_backend()`, senza impegnare risorse hardware.
2. **Esecuzione su hardware reale** (`mode="real_hardware"`): richiede credenziali valide di IBM Quantum Platform.

Entrambe le modalità richiedono il pacchetto `qiskit-ibm-runtime` e, per la connessione al servizio, la variabile d'ambiente `QISKIT_IBM_TOKEN` (iniettata dal container Docker, si veda `docker-compose.yml`). L'esecuzione sull'hardware reale è protetta da un flag esplicito (`RUN_ON_REAL_HARDWARE`) per evitare di consumare inavvertitamente tempo di QPU a pagamento.

**Nota sulla dimensione del training set**: per calcolare la matrice di kernel, `FidelityQuantumKernel` invia **un'unica chiamata al sampler**, contenente una coppia di circuiti per ciascuna coppia di campioni di addestramento (una lista di PUB, si veda `ComputeUncompute._run` e `src/quantum/qsvc_model.py`), non un circuito o un job separato per coppia. In simulazione ideale (`StatevectorSampler`) questo non è un problema (verificato: l'intero training set di Breast Cancer Wisconsin, 455 campioni, ~103.000 coppie, richiede un picco di memoria di solo ~785 MB). In modalità `noisy_simulation`, invece, `AerSampler` ha un'impronta di memoria per coppia molto più ripida, e un'unica chiamata con centinaia di campioni può esaurire la RAM disponibile prima di completare il calcolo; su hardware reale, inoltre, una singola chiamata così grande può eccedere i limiti per job del servizio IBM Quantum Platform.

La soluzione adottata è `max_circuits_per_job` (parametro nativo di `FidelityQuantumKernel`, inoltrato da `train_and_evaluate_qsvc`/`build_qsvc`): suddivide la stessa, unica chiamata logica in più chiamate più piccole ("chunk"), riducendo drasticamente il picco di memoria per chiamata a fronte di un modesto overhead aggiuntivo, invece di dover ridurre il numero di coppie effettivamente calcolate. `QSVC_MAX_CIRCUITS_PER_JOB`, definito in `config.py` insieme agli altri iperparametri sperimentali, imposta questo parametro per le celle `noisy_simulation` e `real_hardware` di questo notebook. `HARDWARE_TRAIN_SAMPLE_SIZE`/`HARDWARE_TEST_SAMPLE_SIZE` sottocampionano comunque il training/test set per queste due modalità, a scopo dimostrativo e per contenere i tempi di esecuzione, in particolare il tempo di attesa in coda su hardware reale, ma grazie al chunking non è più necessario limitarli in modo così aggressivo come in assenza di questo meccanismo: è una limitazione pratica aggiuntiva rispetto al "sottoinsieme rappresentativo di configurazioni" già descritto nella sezione "Strategia progressiva di valutazione" (Capitolo 3), qui riferita al numero di campioni all'interno di una singola configurazione.

In [ ]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd

from config import (
    DATASET_CONFIGS, RANDOM_STATE, TEST_SIZE, VQC_MAXITER,
    HARDWARE_TRAIN_SAMPLE_SIZE, HARDWARE_TEST_SAMPLE_SIZE, QSVC_MAX_CIRCUITS_PER_JOB,
    QSVC_REAL_HARDWARE_MAX_CIRCUITS_PER_JOB,
)
from src.execution.backend_manager import build_noise_model_from_backend, get_pass_manager
from src.data.preprocessing import stratified_subsample
from src.pipeline import load_and_preprocess_dataset, train_and_evaluate_qsvc, train_and_evaluate_vqc

TABLES_DIR = Path.cwd().parent / "results" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Sottoinsieme rappresentativo di configurazioni da eseguire su hardware
# reale (sezione "Strategia progressiva di valutazione", Capitolo 3). A
# differenza di HARDWARE_TRAIN_SAMPLE_SIZE/HARDWARE_TEST_SAMPLE_SIZE e
# QSVC_MAX_CIRCUITS_PER_JOB (iperparametri sperimentali, centralizzati in
# config.py), questi sono controlli di esecuzione specifici di questo
# notebook e restano definiti localmente.
HARDWARE_SUBSET = ["breast_cancer"]
RUN_ON_REAL_HARDWARE = True  # impostare a True solo consapevolmente
# Il VQC su hardware reale invia una chiamata al sampler per ciascuna
# valutazione della funzione di costo (fino a VQC_MAXITER per dataset),
# a differenza del QSVC che le raggruppa in poche decine di chiamate
# tramite QSVC_MAX_CIRCUITS_PER_JOB: resta disattivato per la prima
# campagna sui tre dataset, per valutare tempi di esecuzione e consumo
# della quota QPU del solo QSVC prima di impegnare ulteriore tempo di
# coda/QPU sul VQC.
RUN_VQC_ON_REAL_HARDWARE = False
BACKEND_NAME = None  # es. "ibm_kyiv"; se None, viene selezionato il backend meno occupato

In [2]:
try:
    from qiskit_ibm_runtime import QiskitRuntimeService

    service = QiskitRuntimeService(
        channel=os.environ.get("QISKIT_IBM_CHANNEL", "ibm_quantum_platform"),
        token=os.environ.get("QISKIT_IBM_TOKEN"),
    )
    calibration_backend = service.least_busy(operational=True, simulator=False)
    noise_model = build_noise_model_from_backend(calibration_backend)
    print(f"Modello di rumore costruito a partire dal backend: {calibration_backend.name} "
          f"({calibration_backend.num_qubits} qubit)")
except Exception as exc:  # credenziali assenti o servizio non raggiungibile
    service = None
    noise_model = None
    print(f"Connessione a IBM Quantum Platform non disponibile: {exc}")
    print("Le celle di simulazione con rumore ed esecuzione hardware verranno saltate.")

qiskit_runtime_service._discover_account:WARNING:2026-09-10 09:22:00,888: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-09-10 09:22:05,095: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: thesis-l8. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-09-10 09:22:05,597: Loading instance: thesis-l8, plan: open
qiskit_runtime_service.backends:WARNING:2026-09-10 09:22:08,400: Using instance: thesis-l8, plan: open


Modello di rumore costruito a partire dal backend: ibm_kingston (156 qubit)


## Simulazione con modello di rumore

In [ ]:
noisy_rows = []

if noise_model is not None:
    for name in HARDWARE_SUBSET:
        cfg = DATASET_CONFIGS[name]
        X_train, X_test, y_train, y_test, _pca = load_and_preprocess_dataset(
            name, cfg["n_components"], test_size=TEST_SIZE, random_state=RANDOM_STATE,
        )
        # Sottocampionamento stratificato (non un semplice troncamento
        # posizionale): preserva le proporzioni tra classi anche a
        # dimensioni ridotte (si veda src/data/preprocessing.py).
        X_train, y_train = stratified_subsample(
            X_train, y_train, HARDWARE_TRAIN_SAMPLE_SIZE, random_state=RANDOM_STATE,
        )
        X_test, y_test = stratified_subsample(
            X_test, y_test, HARDWARE_TEST_SAMPLE_SIZE, random_state=RANDOM_STATE,
        )

        # Il pass manager è costruito per-dataset: dipende da n_components
        # (il numero di qubit logici usati), non solo dal backend
        # (sezione "Gestione dei backend di esecuzione", Capitolo 4 - la
        # coupling map viene ridotta a soli n_qubit fisici mutuamente
        # connessi, invece dell'intero registro del backend, tipicamente
        # 127+ qubit sui dispositivi IBM attuali).
        pass_manager = get_pass_manager("noisy_simulation", backend=calibration_backend,
                                         n_qubits=cfg["n_components"])

        qsvc_result = train_and_evaluate_qsvc(
            cfg["n_components"], cfg["feature_map_reps"], cfg["entanglement"],
            X_train, y_train, X_test, y_test,
            sampler_mode="noisy_simulation", noise_model=noise_model,
            pass_manager=pass_manager, random_state=RANDOM_STATE,
            max_circuits_per_job=QSVC_MAX_CIRCUITS_PER_JOB,
        )
        noisy_rows.append({"dataset": name, **qsvc_result})

        vqc_result, _cost_history = train_and_evaluate_vqc(
            cfg["n_components"], cfg["feature_map_reps"], cfg["ansatz_reps"],
            cfg["entanglement"], X_train, y_train, X_test, y_test,
            maxiter=VQC_MAXITER, sampler_mode="noisy_simulation", noise_model=noise_model,
            pass_manager=pass_manager, random_state=RANDOM_STATE,
        )
        noisy_rows.append({"dataset": name, **vqc_result})

    noisy_df = pd.DataFrame(noisy_rows).drop(columns="confusion_matrix")
    noisy_df.to_csv(TABLES_DIR / "results_noisy_simulation.csv", index=False)
    display(noisy_df)
else:
    print("Cella saltata: nessun modello di rumore disponibile.")

## Esecuzione su hardware quantistico reale

**Attenzione**: questa cella invia effettivamente i circuiti a un dispositivo quantistico reale tramite IBM Quantum Platform e consuma tempo di QPU. Viene eseguita solo se `RUN_ON_REAL_HARDWARE = True` ed è stato possibile connettersi al servizio.

Per la prima campagna sui tre dataset, il VQC è disattivato di default (`RUN_VQC_ON_REAL_HARDWARE = False`): a differenza del QSVC, che raggruppa il calcolo del kernel in poche decine di chiamate tramite `QSVC_MAX_CIRCUITS_PER_JOB`, il VQC invia una chiamata al sampler per ciascuna valutazione della funzione di costo (fino a `VQC_MAXITER` per dataset), con un numero di job e un tempo di attesa in coda potenzialmente molto più elevati. Si esegue quindi prima il solo QSVC su tutti e tre i dataset, per valutare tempi di esecuzione effettivi e consumo della quota QPU disponibile, prima di decidere se e con quale `maxiter` includere anche il VQC.

In [ ]:
hardware_rows = []

if RUN_ON_REAL_HARDWARE and service is not None:
    target_backend = service.backend(BACKEND_NAME) if BACKEND_NAME else calibration_backend
    backend_name = target_backend.name

    for name in HARDWARE_SUBSET:
        cfg = DATASET_CONFIGS[name]
        X_train, X_test, y_train, y_test, _pca = load_and_preprocess_dataset(
            name, cfg["n_components"], test_size=TEST_SIZE, random_state=RANDOM_STATE,
        )
        # Sottocampionamento stratificato (non un semplice troncamento
        # posizionale): preserva le proporzioni tra classi anche a
        # dimensioni ridotte (si veda src/data/preprocessing.py).
        X_train, y_train = stratified_subsample(
            X_train, y_train, HARDWARE_TRAIN_SAMPLE_SIZE, random_state=RANDOM_STATE,
        )
        X_test, y_test = stratified_subsample(
            X_test, y_test, HARDWARE_TEST_SAMPLE_SIZE, random_state=RANDOM_STATE,
        )

        pass_manager = get_pass_manager("real_hardware", backend=target_backend,
                                         n_qubits=cfg["n_components"])

        qsvc_result = train_and_evaluate_qsvc(
            cfg["n_components"], cfg["feature_map_reps"], cfg["entanglement"],
            X_train, y_train, X_test, y_test,
            sampler_mode="real_hardware", backend_name=backend_name,
            pass_manager=pass_manager, random_state=RANDOM_STATE,
            max_circuits_per_job=QSVC_REAL_HARDWARE_MAX_CIRCUITS_PER_JOB,
        )
        hardware_rows.append({"dataset": name, "backend": backend_name, **qsvc_result})

        if RUN_VQC_ON_REAL_HARDWARE:
            vqc_result, _cost_history = train_and_evaluate_vqc(
                cfg["n_components"], cfg["feature_map_reps"], cfg["ansatz_reps"],
                cfg["entanglement"], X_train, y_train, X_test, y_test,
                maxiter=VQC_MAXITER, sampler_mode="real_hardware", backend_name=backend_name,
                pass_manager=pass_manager, random_state=RANDOM_STATE,
            )
            hardware_rows.append({"dataset": name, "backend": backend_name, **vqc_result})

    hardware_df = pd.DataFrame(hardware_rows).drop(columns="confusion_matrix")
    hardware_df.to_csv(TABLES_DIR / "results_real_hardware.csv", index=False)
    display(hardware_df)
else:
    print("Cella saltata: impostare RUN_ON_REAL_HARDWARE = True e disporre di credenziali valide "
          "per eseguire questa sezione.")

qiskit_runtime_service._discover_account:WARNING:2026-09-10 09:22:22,118: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-09-10 09:22:24,951: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: thesis-l8. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-09-10 09:22:24,953: Using instance: thesis-l8, plan: open
/home/thesis-user/.local/lib/python3.12/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py:1270: UserWarning: This instance has met its usage limit. Workloads will not r